In [ ]:
class RelationNormalizer:

    def __init__(self, schema_path):
        self.schema = self.load_schema(schema_path)


    def normalize(self, raw_data, document_id):

        id_mapping = {}

        entities = self.normalize_entities(
            raw_data["entities"],
            id_mapping
        )

        relationships = self.normalize_relationships(
            raw_data["relationships"],
            id_mapping
        )

        return {
            "document_id": document_id,
            "schema_version": "1.0",
            "entities": entities,
            "relationships": relationships
        }

In [8]:
import json
from os import path

path = "schema_registry.json"
with open(path, "r", encoding="utf-8") as file:
    inp = json.load(file)

print(inp)

{'schema_version': '1.0', 'entity_types': ['Person', 'Organization', 'Location', 'Facility', 'Event', 'Product', 'Work', 'Publication', 'Technology', 'Concept', 'Law', 'Language', 'Award', 'Time', 'Number'], 'relationship_types': {'WORKS_AT': {'source_types': ['Person'], 'target_types': ['Organization']}, 'FOUNDED': {'source_types': ['Person', 'Organization'], 'target_types': ['Organization']}, 'MEMBER_OF': {'source_types': ['Person', 'Organization'], 'target_types': ['Organization']}, 'HEAD_OF': {'source_types': ['Person'], 'target_types': ['Organization']}, 'EDUCATED_AT': {'source_types': ['Person'], 'target_types': ['Organization']}, 'REPRESENTS': {'source_types': ['Person'], 'target_types': ['Organization']}, 'SUBSIDIARY_OF': {'source_types': ['Organization'], 'target_types': ['Organization']}, 'PART_OF': {'source_types': ['Organization', 'Location', 'Facility', 'Event', 'Product', 'Work', 'Publication', 'Technology', 'Concept'], 'target_types': ['Organization', 'Location', 'Facili

In [10]:
allowed_relations = set(inp.get("relationship_types", {}).keys())
allowed_relations

{'AUTHORED_BY',
 'BORN_IN',
 'CAPITAL_OF',
 'CAUSES',
 'CITIZEN_OF',
 'COMPARES_WITH',
 'COMPETITOR_OF',
 'CREATED_BY',
 'DESTINATION',
 'DEVELOPED_BY',
 'DIED_IN',
 'DIRECTED_BY',
 'EDUCATED_AT',
 'ENDED_AT',
 'EVALUATED_BY',
 'FEATURE_OF',
 'FORMED_IN',
 'FOUNDED',
 'HEADQUARTERED_IN',
 'HEAD_OF',
 'INSTANCE_OF',
 'INVOLVED_IN',
 'LOCATED_IN',
 'MANUFACTURED_BY',
 'MEMBER_OF',
 'OCCURRED_AT',
 'OPERATED_BY',
 'ORIGINATES_FROM',
 'OWNS',
 'PARENT_OF',
 'PARTNER_OF',
 'PART_OF',
 'PERFORMED_BY',
 'PRODUCED_BY',
 'PUBLISHED_BY',
 'PUBLISHED_IN',
 'RECEIVED_AWARD',
 'REPRESENTS',
 'SIBLING_OF',
 'SPOUSE_OF',
 'STARTED_AT',
 'SUBCLASS_OF',
 'SUBSIDIARY_OF',
 'TOPIC_OF',
 'USED_FOR',
 'USES',
 'WORKS_AT'}

In [11]:
type(allowed_relations)

set

In [1]:
import json
from copy import deepcopy
from pathlib import Path


class RelationNormalizer:
    """
    Normalize LLM relation extraction output
    into project standard contract.
    """

    def __init__(self, schema_path: str):

        self.schema = self._load_schema(schema_path)

        self.allowed_relations = set(self.schema.get("relationship_types", {}).keys())


    def _load_schema(self, schema_path: str):

        path = Path(schema_path)

        if not path.exists():
            raise FileNotFoundError(f"Schema file not found: {schema_path}")

        with open(path, "r", encoding="utf-8") as file:

            return json.load(file)


    def normalize(self, raw_data: dict, document_id: str) -> dict:
        """
        Main normalization pipeline.
        """

        raw_data = deepcopy(raw_data)

        id_mapping = {}


        entities = self._normalize_entities(raw_data.get("entities", []),id_mapping)


        relationships = self._normalize_relationships(raw_data.get("relationships", []),id_mapping)


        return {

            "document_id": document_id,

            "schema_version": "1.0",

            "entities": entities,

            "relationships": relationships
        }



    def _normalize_entities(self, entities: list, id_mapping: dict) -> list:


        normalized_entities = []


        for index, entity in enumerate(entities,start=1):

            old_id = entity.get("id")

            new_id = f"e{index}"


            if old_id:
                id_mapping[old_id] = new_id


            normalized_entities.append({

                "id": new_id,

                "name": entity.get(
                    "name",
                    ""
                ),

                "type": entity.get(
                    "type",
                    "UNKNOWN"
                ),

                "properties":
                    entity.get(
                        "properties",
                        {}
                    )
            })


        return normalized_entities



    def _normalize_relationships(
        self,
        relationships: list,
        id_mapping: dict ) -> list:


        normalized_relationships = []


        for relation in relationships:


            source = relation.get("source")

            target = relation.get("target")

            relation_type = relation.get("type","UNKNOWN")


            normalized_relationships.append({

                "source":
                    id_mapping.get(source, source),

                "target":
                    id_mapping.get(target, target),


                "type":
                    relation_type,


                "new":
                    relation_type
                    not in
                    self.allowed_relations,


                "properties":
                    relation.get("properties", {})
            })


        return normalized_relationships

In [14]:
from relation_normalizer import RelationNormalizer
import json


normalizer = RelationNormalizer("schema_registry.json")

# خواندن خروجی خام LLM
with open("response.json","r", encoding="utf-8") as file: 
    raw_data = json.load(file)

result = normalizer.normalize(
    raw_data,
    "doc_001"
)


print(result)

{'document_id': 'doc_001', 'schema_version': '1.0', 'entities': [{'id': 'e1', 'name': 'شرکت نوآوران سبز', 'type': 'Organization', 'properties': {'established_year': '۱۳۸۵'}}, {'id': 'e2', 'name': 'دکتر کریمی', 'type': 'Person', 'properties': {'role': 'رئیس هیئت\u200cمدیره', 'previous_role': 'استاد دانشکده مهندسی برق دانشگاه تهران'}}, {'id': 'e3', 'name': 'دانشگاه صنعتی شریف', 'type': 'Organization', 'properties': {}}, {'id': 'e4', 'name': 'قرارداد همکاری', 'type': 'Event', 'properties': {'date': '۱۰ آبان ۱۴۰۲', 'subject': 'پژوهش در زمینه انرژی\u200cهای تجدیدپذیر و توسعه باتری\u200cهای لیتیوم\u200cیونی'}}, {'id': 'e5', 'name': 'دانشگاه تهران', 'type': 'Organization', 'properties': {}}, {'id': 'e6', 'name': 'مهندس رضویان', 'type': 'Person', 'properties': {'role': 'معاون پژوهشی شرکت نوآوران سبز'}}, {'id': 'e7', 'name': 'نشریه بین\u200cالمللی انرژی', 'type': 'Organization', 'properties': {}}, {'id': 'e8', 'name': 'آلمان', 'type': 'Location', 'properties': {}}, {'id': 'e9', 'name': 'صنایع ا

In [1]:
from relation_normalizer import RelationNormalizer


raw_data = {

    "entities": [

        {
            "id":"dr_karimi",
            "name":"دکتر کریمی",
            "type":"Person",
            "properties":{
                "role":"رئیس هیئت مدیره"
            }
        },

        {
            "id":"company_1",
            "name":"شرکت نوآوران سبز",
            "type":"Organization",
            "properties":{}
        }

    ],


    "relationships":[

        {
            "source":"dr_karimi",
            "target":"company_1",
            "type":"FOUNDED",
            "properties":{
                "year":"۱۳۸۵",
                "source_text":
                "شرکت نوآوران سبز توسط دکتر کریمی تأسیس شد."
            }
        }

    ]

}



normalizer = RelationNormalizer(
    "schema_registry.json"
)


result = normalizer.normalize(
    raw_data,
    "doc_001"
)


print(result)

{'document_id': 'doc_001', 'entities': [{'source_id': 'dr_karimi', 'name': 'دکتر کریمی', 'type': 'Person', 'properties': {'role': 'رئیس هیئت مدیره'}}, {'source_id': 'company_1', 'name': 'شرکت نوآوران سبز', 'type': 'Organization', 'properties': {}}], 'relationships': [{'source': 'dr_karimi', 'target': 'company_1', 'type': 'FOUNDED', 'schema_exists': True, 'properties': {'year': '۱۳۸۵', 'source_text': 'شرکت نوآوران سبز توسط دکتر کریمی تأسیس شد.'}}]}


In [2]:
import hazm